In [ ]:
###
# 1. 環境のセットアップ
###

import sys
import os
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

ROOT_PATH = Path('/content/drive/MyDrive/cnn-hands-on')
if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

os.chdir(ROOT_PATH)

# 日本語フォント対応
!pip install -q japanize-matplotlib
import japanize_matplotlib

print(f"✅ 環境セットアップ完了！現在のディレクトリ: {Path.cwd()}")

In [ ]:
###
# 2. 必要なライブラリのインポート
###

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用するデバイス: {device}")

# 3. MNISTデータの読み込み

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

images, labels = next(iter(train_loader))
print(f"画像: {images.shape}, ラベル: {labels.shape}")

In [ ]:
# サンプル画像の表示
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.set_title(f"ラベル: {labels[i].item()}", fontsize=12)
    ax.axis('off')
plt.suptitle('MNISTサンプル画像', fontsize=14)
plt.tight_layout()
plt.show()

# 4. 損失関数を体験する

In [ ]:
criterion = nn.CrossEntropyLoss()

# 良い予測
good_output = torch.tensor([[5.0, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]])
label = torch.tensor([0])
print(f"良い予測の損失: {criterion(good_output, label).item():.4f}")

# 悪い予測
bad_output = torch.tensor([[0.1, 0.1, 0.1, 0.1, 0.1, 5.0, 0.1, 0.1, 0.1, 0.1]])
print(f"悪い予測の損失: {criterion(bad_output, label).item():.4f}")

# ランダム予測
random_output = torch.randn(1, 10)
print(f"ランダム予測の損失: {criterion(random_output, label).item():.4f}")

# 5. CNNモデルの定義

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN().to(device)
print(model)

# 6. 学習ループの実装

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
num_epochs = 5
train_loss_list = []
test_loss_list = []
test_acc_list = []

print("学習開始!")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    epoch_train_loss = running_loss / len(train_loader)
    train_loss_list.append(epoch_train_loss)

    model.eval()
    running_test_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_test_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_test_loss = running_test_loss / len(test_loader)
    epoch_test_acc = 100 * correct / total
    test_loss_list.append(epoch_test_loss)
    test_acc_list.append(epoch_test_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Train Loss: {epoch_train_loss:.4f} | "
          f"Test Loss: {epoch_test_loss:.4f} | "
          f"Test Acc: {epoch_test_acc:.2f}%")

print("\n学習完了!")

# 7. 学習結果の可視化

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, num_epochs+1), train_loss_list, label='Train Loss', marker='o', color='blue')
ax1.plot(range(1, num_epochs+1), test_loss_list, label='Test Loss', marker='o', color='orange')
ax1.set_title('Loss（損失）の推移')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(range(1, num_epochs+1), test_acc_list, label='Test Accuracy', marker='o', color='green')
ax2.set_title('Accuracy（正答率）の推移')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

# 8. 推論

In [ ]:
model.eval()
test_images, test_labels = next(iter(test_loader))
test_images = test_images[:10].to(device)
test_labels = test_labels[:10]

with torch.no_grad():
    outputs = model(test_images)
    _, predicted = torch.max(outputs, 1)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(test_images[i].cpu().squeeze(), cmap='gray')
    pred = predicted[i].item()
    true = test_labels[i].item()
    color = 'green' if pred == true else 'red'
    ax.set_title(f"予測: {pred} (正解: {true})", fontsize=11, color=color)
    ax.axis('off')

plt.suptitle('推論結果（緑=正解、赤=不正解）', fontsize=14)
plt.tight_layout()
plt.show()